# 01 - Bitext dataset exploration

Run from the project environment. This notebook audits raw data without training a model or changing labels. The standard-library pipeline also runs without Jupyter.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Open this notebook from the project or notebooks directory")
sys.path.insert(0, str(ROOT))
from src.data.inspect import inspect_csv, write_report

## Inspect the raw dataset

First run `python -m src.data.download` from the project root. The pinned source and file hashes are saved in `data/raw/manifest.json`.

In [2]:
summary = inspect_csv(ROOT / "data/raw/bitext.csv")
{k: summary[k] for k in ["rows", "columns", "missing_by_column", "word_count"]}

{'rows': 26872,
 'columns': ['flags', 'instruction', 'category', 'intent', 'response'],
 'missing_by_column': {'flags': 0,
  'instruction': 0,
  'category': 0,
  'intent': 0,
  'response': 0},
 'word_count': {'min': 1, 'median': 9.0, 'mean': 8.69, 'max': 16}}

In [3]:
summary["category_counts"], summary["intent_counts"]

({'ACCOUNT': 5986,
  'CANCEL': 950,
  'CONTACT': 1999,
  'DELIVERY': 1994,
  'FEEDBACK': 1997,
  'INVOICE': 1999,
  'ORDER': 3988,
  'PAYMENT': 1998,
  'REFUND': 2992,
  'SHIPPING': 1970,
  'SUBSCRIPTION': 999},
 {'cancel_order': 998,
  'change_order': 997,
  'change_shipping_address': 973,
  'check_cancellation_fee': 950,
  'check_invoice': 1000,
  'check_payment_methods': 999,
  'check_refund_policy': 997,
  'complaint': 1000,
  'contact_customer_service': 1000,
  'contact_human_agent': 999,
  'create_account': 997,
  'delete_account': 995,
  'delivery_options': 995,
  'delivery_period': 999,
  'edit_account': 1000,
  'get_invoice': 999,
  'get_refund': 997,
  'newsletter_subscription': 999,
  'payment_issue': 999,
  'place_order': 998,
  'recover_password': 995,
  'registration_problems': 999,
  'review': 997,
  'set_up_shipping_address': 997,
  'switch_account': 1000,
  'track_order': 995,
  'track_refund': 998})

In [4]:
{k: summary[k] for k in ["exact_duplicate_rows", "duplicate_texts_exact", "duplicate_texts_normalized", "conflicting_normalized_text_groups"]}

{'exact_duplicate_rows': 0,
 'duplicate_texts_exact': 2237,
 'duplicate_texts_normalized': 2598,
 'conflicting_normalized_text_groups': 0}

## Visualize distributions

The generated HTML contains category, intent and word-length distribution bars. Counts exclude rows missing required input or target values.

In [5]:
write_report(summary, ROOT / "reports")
from IPython.display import HTML, display
display(HTML((ROOT / "reports/eda_report.html").read_text(encoding="utf-8")))

ACCOUNT,,"5,986"
CANCEL,,950
CONTACT,,"1,999"
DELIVERY,,"1,994"
FEEDBACK,,"1,997"
INVOICE,,"1,999"
ORDER,,"3,988"
PAYMENT,,"1,998"
REFUND,,"2,992"
SHIPPING,,"1,970"
SUBSCRIPTION,,999


## Findings and next decisions

The initial audit found 26,872 rows, 11 categories and 27 intents. There are no missing fields. There are 2,598 repeated messages beyond the first occurrence after case/whitespace normalization, without conflicting labels.

Before training: deduplicate, create reproducible splits stratified by intent, and assert no normalized message overlap. Inspect template similarity separately; exact deduplication does not eliminate paraphrase leakage. Use only `instruction` as input. Each intent maps to one category in this revision, allowing category derivation from intent.

Questions: Why do full-row duplicate counts differ from message duplicate counts? Why is class balance different for intents and categories? Why should we fit TF-IDF only on the training split?